In [ ]:
#Setting up the notebook with important libraries
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from keras.layers import Dense
from keras.models import Sequential
import numpy as np
import matplotlib.pyplot as plt
from time import time
import os

from google.colab import drive
drive.mount("/content/drive")

"""""The goal of this assignment is to develop deep learning models (RESNET50, VGG16, etc) that are able to predict with a
high level of accuracy which dental procedure a patient received (implant, filling, impacted tooth, cavity) based on images of
the relevant dental diagnosis. These dental images will work in conjunction with our 3 datasets (train, test, valid) when developing
our deep learning models. We begin with importing the necessary python libraries that will provide us the functionality we desire to
succesfully accomplish this task. TensorFlow allows us to utilize neural networks in conjunction with our data, allowing us to
build the models we need for this endeavor. Keras has been imported as well to access the numerous functionalies it has as its
disposal. We are able to effectively harness the Keras library as its layer-based structure allows us to design neural networks
step by step, adding components such as Dense layers for classification or convolutional layers for image analysis.
This flexibility is essential for our dental project as it lets us train models to accurately identify patterns in images and
predict different procedures with precision. Keras makes it easier to focus on creating an effective model while handling
complex computations in the background. Finally, the Google Drive is mounted to the Colab notebook, allowing access to files
stored in our Drive (useful for loading datasets and images - train, test and valid)."""""


In [ ]:
from google.colab import drive
import zipfile
import os

# Step 1: Mount Google Drive
drive.mount('/content/drive')

# Step 2: Define the file path
zip_file_path = '/content/drive/My Drive/dataset.zip'  # Replace with your file's path

# Step 3: Check if the file exists
if os.path.exists(zip_file_path):
    print("File found!")
    # Step 4: Extract the file
    extracted_dir = '/content/drive/My Drive/final project'  # Folder to extract files
    os.makedirs(extracted_dir, exist_ok=True)

    with zipfile.ZipFile(zip_file_path, 'r') as zip_ref:
        zip_ref.extractall(extracted_dir)
    print(f"Files extracted to: {extracted_dir}")
else:
    print("File not found!")

    """""During this step we are accessing our Google Drive (using the relevant file path) to extract the zip file containing our
project's dataset and images, which are to be peered through later during data analysis"""""


In [ ]:
#1Data cleaning
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.impute import SimpleImputer
import os
from google.colab import drive
drive.mount("/content/drive")

# Function for data cleaning and organizing
def clean_and_organize_data(df):
    # Step 1: Handle missing values
    print("Handling missing values...")

    # Impute numerical columns with the median value
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    imputer_num = SimpleImputer(strategy='median')
    df[num_cols] = imputer_num.fit_transform(df[num_cols])

    # Impute categorical columns with the most frequent value (mode)
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    imputer_cat = SimpleImputer(strategy='most_frequent')
    df[cat_cols] = imputer_cat.fit_transform(df[cat_cols])

    # Step 2: Remove duplicates
    print("Removing duplicates...")
    df = df.drop_duplicates()

    # Step 3: Handle outliers (using IQR method for numerical columns)
    def remove_outliers(df, column_name):
        Q1 = df[column_name].quantile(0.25)
        Q3 = df[column_name].quantile(0.75)
        IQR = Q3 - Q1
        lower_bound = Q1 - 1.5 * IQR
        upper_bound = Q3 + 1.5 * IQR
        return df[(df[column_name] >= lower_bound) & (df[column_name] <= upper_bound)]

    print("Handling outliers...")
    for col in num_cols:
        df = remove_outliers(df, col)

    # Step 4: Correct data types (e.g., ensure datetime columns are datetime type)
    if 'appointment_date' in df.columns:
        df['appointment_date'] = pd.to_datetime(df['appointment_date'], errors='coerce')

    # Convert categorical columns to 'category' dtype (for efficiency)
    for col in cat_cols:
        df[col] = df[col].astype('category')

    # Step 5: Standardize column names
    print("Standardizing column names...")
    df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')

    # Step 6: Feature Engineering (example: extract day of the week from 'appointment_date')
    if 'appointment_date' in df.columns:
        df['appointment_day'] = df['appointment_date'].dt.day_name()

    # Step 7: Encoding categorical variables (Label Encoding or One-Hot Encoding)
    print("Encoding categorical columns...")
    label_encoder = LabelEncoder()
    for col in cat_cols:
        df[col] = label_encoder.fit_transform(df[col])

    # Step 8: Normalize numerical columns (feature scaling)
    print("Normalizing numerical columns...")
    scaler = StandardScaler()
    df[num_cols] = scaler.fit_transform(df[num_cols])

    # Step 9: Summary of the cleaning process
    print("\nData Cleaning Summary:")
    print(f"Shape of data after cleaning: {df.shape}")
    print(f"Missing values after cleaning: {df.isnull().sum()}")
    print(f"First few rows of the cleaned data:\n{df.head()}")

    return df

# Function to process and clean multiple datasets from different folders
def process_data_folders(base_folder):
    # Define paths for the train, validate, and test datasets
    train_file_path = os.path.join(base_folder, 'train', 'train.csv')
    validate_file_path = os.path.join(base_folder, 'validate', 'validat.csv')
    test_file_path = os.path.join(base_folder, 'test', 'test.csv')

    # List of paths to process
    file_paths = [train_file_path, validate_file_path, test_file_path]

    # Process each file
    for file_path in file_paths:
        if os.path.exists(file_path):
            print(f"\nProcessing file: {file_path}")
            # Step 1: Load the dataset
            df = pd.read_csv(file_path)

            # Step 2: Clean the data
            cleaned_df = clean_and_organize_data(df)

            # Step 3: Save the cleaned dataset
            cleaned_file_path = file_path.replace('.csv', '_cleaned.csv')
            cleaned_df.to_csv(cleaned_file_path, index=False)
            print(f"Cleaned data saved to: {cleaned_file_path}")
        else:
            print(f"File not found: {file_path}")

# Main execution
base_folder = '/content/drive/My Drive/final project'  # Adjust this path to your project folder
process_data_folders(base_folder)


"""""We proceed to our next step which is to clean our dataset. We begin by importing the necessary libaries needed for data
cleansing, the most important of which incldue numpy (data manipulation and numerical operations) and scikit-learn (containing
modules to replace missing values, standardize numerical data, etc.). In our case we harnessed and utilized these libraries
with a specific purpose. The first was to replace any missing numerical values found in features such as xmin or y min
(which in this case were filled with that feature's mean). Meanwhile, categorical values (e.g. procedure types like "Implant" or "Cavity"),
are replaced with the mode or most frequently occurring instance. We have also proceeded with removing any duplicate values
in terms of file name. Interquartile Range (IQR) was implemented to avoid any outliers that would significantly skew our data.
We also proceeded to remove any potential syntax errors by standardizing all the feature columns. Before our final step of data
loading the new cleaned versions we also made sure to encode any features from categorical values to numerical
(e.g., "Cavity" → 0, "Implant" → 1), which in turn transformed our dataset into one that is more suitable for the models we
are creating through machine learning.""""


In [ ]:
#3-3. Exploratory Data Analysis
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from google.colab import drive
drive.mount("/content/drive")
# Function to perform EDA on a single dataset
def perform_eda(df, dataset_name):
    print(f"\n--- EDA for {dataset_name} ---")

    # Step 1: Basic Info about the Data
    print(f"\nBasic Information for {dataset_name}:")
    print(df.info())

    # Step 2: Statistical Summary of the Numerical Columns
    print(f"\nStatistical Summary for {dataset_name}:")
    print(df.describe())

    # Step 3: Check for Missing Values
    print(f"\nMissing Values in {dataset_name}:")
    print(df.isnull().sum())

    # Step 4: Visualizations
    # 4.1 Distribution of Numerical Features (Histograms)
    num_cols = df.select_dtypes(include=['float64', 'int64']).columns
    for col in num_cols:
        plt.figure(figsize=(6, 4))
        sns.histplot(df[col], kde=True)
        plt.title(f'Distribution of {col} in {dataset_name}')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.show()

    # 4.2 Correlation Heatmap for Numerical Features
    plt.figure(figsize=(10, 8))
    correlation_matrix = df[num_cols].corr()
    sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
    plt.title(f'Correlation Heatmap for {dataset_name}')
    plt.show()

    # 4.3 Bar Plots for Categorical Variables
    cat_cols = df.select_dtypes(include=['object', 'category']).columns
    for col in cat_cols:
        plt.figure(figsize=(6, 4))
        sns.countplot(x=col, data=df)
        plt.title(f'Count Plot for {col} in {dataset_name}')
        plt.xlabel(col)
        plt.ylabel('Count')
        plt.xticks(rotation=45)
        plt.show()

    # 4.4 Pairplot to check relationships between numerical features (optional for small datasets)
    if len(num_cols) > 1:  # Only do this if there are at least 2 numerical columns
        sns.pairplot(df[num_cols], plot_kws={'alpha': 0.5})
        plt.suptitle(f'Pairplot of Numerical Features in {dataset_name}', y=1.02)
        plt.show()

# Function to process and perform EDA for multiple datasets from different folders
def perform_eda_for_folders(base_folder):
    # Define paths for the train, validate, and test datasets
    train_file_path = os.path.join(base_folder, 'train', 'train.csv')
    validate_file_path = os.path.join(base_folder, 'valid', 'validate.csv')
    test_file_path = os.path.join(base_folder, 'test', 'test.csv')

    # List of paths to process
    file_paths = [(train_file_path, 'train'),
                  (validate_file_path, 'validate'),
                  (test_file_path, 'test')]

    # Process each file
    for file_path, dataset_name in file_paths:
        if os.path.exists(file_path):
            print(f"\nProcessing EDA for {dataset_name} dataset...")
            # Step 1: Load the dataset
            df = pd.read_csv(file_path)

            # Step 2: Perform EDA
            perform_eda(df, dataset_name)
        else:
            print(f"File not found: {file_path}")

# Main execution
base_folder = '/content/drive/My Drive/final project'  # Adjust this path to your project folder
perform_eda_for_folders(base_folder)

"""""During the Explaratory Data Analysis (EDA) step, we utilize the function to produce visual summaries of key numerical
components within our dental imaging dataset. The first piece of information provided through EDA is a statistical
summary displaying a visual snapshot of descriptive statistics (mean, min, max) related to each of our 3 datasets
(train, test, valid). These are useful when observing at a quick glance whether these values are in line with our expecations.
The code follows up with providing us the missing values in each column. As observed once we completed the step to clean our data
we were no longer left with any missing values once they were replaced, ensuring consistency. We proceeded to plot the
distribution of each numerical column to visualize its spread and detect skewness or anomalies (which could possibly disrupt the
model'sperformance or provide us with inaccurate information). A correlation heatmap was created to display the covariability
between the numerical featues in our dataset. This aided our group with removing redundant features that  had limited to no impact
on our dependent variable (dental procedure classification). We were also able to observe which features were closely related,
providing further context for our findings. A correlation heat map also helped in discovering features with covariability which
in turn could distort the predictions coming from our deep learning models. Though of less importance for models employing
deep learning, removing highly correlated features can improve model generalization and training efficiency.""""

In [ ]:
#VGG16 Model:
import os
import tensorflow as tf
from google.colab import drive
from tensorflow.keras.applications import VGG16
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import regularizers

# Mount Google Drive
drive.mount("/content/drive")

# Path to your dataset (update this to your own dataset location)
base_dir = '/content/drive/My Drive/final project'  # Adjust path to where your dataset is located

# Directory structure for image dataset
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'valid')
test_dir = os.path.join(base_dir, 'test')

# VGG16 Model:

def create_vgg16_model(input_shape, num_classes):
    # Load VGG16 model pre-trained on ImageNet without the top classification layers
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)

    # Freeze the base layers
    for layer in base_model.layers:
        layer.trainable = False  # Freeze all layers

    # Add custom layers on top
    x = base_model.output
    x = GlobalAveragePooling2D()(x)

    # Dense layer with L2 regularization
    x = Dense(1024, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = Dropout(0.5)(x)  # Dropout to prevent overfitting

    # Final output layer with softmax activation
    predictions = Dense(num_classes, activation='softmax')(x)

    # Create final model
    model = Model(inputs=base_model.input, outputs=predictions)
    return model

# Image Preprocessing with ImageDataGenerator
train_datagen = ImageDataGenerator(
    rescale=1./255,               # Normalize image pixel values to [0, 1]
    rotation_range=40,            # Randomly rotate images
    width_shift_range=0.2,        # Randomly shift images horizontally
    height_shift_range=0.2,       # Randomly shift images vertically
    shear_range=0.2,              # Apply shear transformations
    zoom_range=0.2,               # Random zoom
    horizontal_flip=True,         # Randomly flip images horizontally
    fill_mode='nearest'           # Fill in missing pixels after transformations
)

validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load data using ImageDataGenerator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # Resize images to 224x224 (required for VGG16)
    batch_size=32,
    class_mode='categorical'  # Multi-class classification
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Create the VGG16 model
input_shape = (224, 224, 3)  # Standard input shape for VGG16
num_classes = len(train_generator.class_indices)  # Number of classes (based on the train dataset)

model = create_vgg16_model(input_shape, num_classes)

# Compile the model with gradient clipping
optimizer = Adam(learning_rate=0.0001)  # Low learning rate to prevent overfitting
model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks: Early stopping and ModelCheckpoint
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_vgg16_model.keras', save_best_only=True)

# Train the model
history = model.fit(
    train_generator,
    epochs=5,  # Reduced to 5 epochs for testing purposes
    validation_data=validation_generator,
    callbacks=[early_stopping, model_checkpoint]
)

# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(test_generator)
print(f'Test Loss: {test_loss}, Test Accuracy: {test_acc}')

# Optional: Save the final model
model.save('final_vgg16_model.keras')  # Saving the final model as .keras file format

"""""After outlining our dataset paths, we proceed to setup the environment for our VGG16 deep learning model. We begin with
loading the pre-trained VGG16 model for it to serve as feature extractor by excluding its top (classification) layers as our model
is already proficient at determining general and generic patterns. We proceed to freeze the base layers to ensure their
current capabilities for recognizing patterns and shapes remains intact and that only new layers will be trained
accordingly. While our Dense Layer learns patterns specific to the dental images, our Output Layer uses softmax activation to
predict the procedure type (e.g., cavity, implant). A dropout was also implemented to randomly deactivate neurons during training,
which would in turn improve generalization. To augment our existing images for more effective image analysis we proceeded with
rescaling and resizing our images (optimal 224x224). The model is compiled with Adam optimizer, which efficiently adjusts
weights during training. The categorical crossentropy loss has also been tailored for multi-class classification. Early stopping
and model checkpoints ensure training halts once performance stabilizes, saving the best-performing weights. This would confirm to the group
that VGG16 is managing its goal to predict dental procedures (based on the provided images) at an effective rate. This in turn splits 
the images into 4 classes. Each model is trained for five epochs, balancing time and performance. In conclusion, as VGG16 is a 
model trained using ImageNet (a compository containing millions of images trained using various categories), it is already in a 
good position to be harnessed for our project. Without having to train this model from scratch, we ensure it does not require a 
large dataset and a vast amount of computational resources to accomplish its task. """"

In [ ]:
#Training and prediction of model
import os
import tensorflow as tf
from google.colab import drive
from tensorflow.keras.applications import VGG16, ResNet50
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
from tensorflow.keras import regularizers
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount("/content/drive")

# Path to your dataset (update this to your own dataset location)
base_dir = '/content/drive/My Drive/final project'  # Adjust path to where your dataset is located

# Directory structure for image dataset
train_dir = os.path.join(base_dir, 'train')
validation_dir = os.path.join(base_dir, 'valid')
test_dir = os.path.join(base_dir, 'test')

# VGG16 Model Function
def create_vgg16_model(input_shape, num_classes):
    base_model = VGG16(weights='imagenet', include_top=False, input_shape=input_shape)
    for layer in base_model.layers:
        layer.trainable = False  # Freeze all layers

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = Dropout(0.5)(x)  # Dropout to prevent overfitting
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model

# ResNet50 Model Function
def create_resnet50_model(input_shape, num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)
    for layer in base_model.layers:
        layer.trainable = False  # Freeze all layers

    x = base_model.output
    x = GlobalAveragePooling2D()(x)
    x = Dense(1024, activation='relu', kernel_regularizer=regularizers.l2(0.01))(x)
    x = Dropout(0.5)(x)
    predictions = Dense(num_classes, activation='softmax')(x)

    model = Model(inputs=base_model.input, outputs=predictions)
    return model

# Image Preprocessing with ImageDataGenerator
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=40,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

validation_datagen = ImageDataGenerator(rescale=1./255)
test_datagen = ImageDataGenerator(rescale=1./255)

# Load data using ImageDataGenerator
train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

validation_generator = validation_datagen.flow_from_directory(
    validation_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Create models
input_shape = (224, 224, 3)
num_classes = len(train_generator.class_indices)

vgg_model = create_vgg16_model(input_shape, num_classes)
resnet_model = create_resnet50_model(input_shape, num_classes)

# Compile models with the Adam optimizer
optimizer = Adam(learning_rate=0.0001)

vgg_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])
resnet_model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks for early stopping and model checkpoint
early_stopping = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
model_checkpoint = ModelCheckpoint('best_model.keras', save_best_only=True)

# Train VGG16 model
vgg_history = vgg_model.fit(
    train_generator,
    epochs=5,
    validation_data=validation_generator,
    callbacks=[early_stopping, model_checkpoint]
)

# Train ResNet50 model
resnet_history = resnet_model.fit(
    train_generator,
    epochs=5,
    validation_data=validation_generator,
    callbacks=[early_stopping, model_checkpoint]
)

# Evaluate both models on the test set
vgg_test_loss, vgg_test_acc = vgg_model.evaluate(test_generator)
resnet_test_loss, resnet_test_acc = resnet_model.evaluate(test_generator)

print(f"VGG16 Test Loss: {vgg_test_loss}, Test Accuracy: {vgg_test_acc}")
print(f"ResNet50 Test Loss: {resnet_test_loss}, Test Accuracy: {resnet_test_acc}")

# Predict the results on the test data
vgg_predictions = vgg_model.predict(test_generator)
resnet_predictions = resnet_model.predict(test_generator)

# Convert predictions to class labels
vgg_pred_labels = np.argmax(vgg_predictions, axis=1)
resnet_pred_labels = np.argmax(resnet_predictions, axis=1)
true_labels = test_generator.classes

# Classification Reports and Confusion Matrices
vgg_class_report = classification_report(true_labels, vgg_pred_labels, target_names=test_generator.class_indices.keys())
resnet_class_report = classification_report(true_labels, resnet_pred_labels, target_names=test_generator.class_indices.keys())

print("VGG16 Classification Report:\n", vgg_class_report)
print("ResNet50 Classification Report:\n", resnet_class_report)

# Confusion Matrices
vgg_conf_matrix = confusion_matrix(true_labels, vgg_pred_labels)
resnet_conf_matrix = confusion_matrix(true_labels, resnet_pred_labels)

# Plot confusion matrices
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].matshow(vgg_conf_matrix, cmap='Blues')
axes[0].set_title('VGG16 Confusion Matrix')
axes[1].matshow(resnet_conf_matrix, cmap='Blues')
axes[1].set_title('ResNet50 Confusion Matrix')

for ax in axes:
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticks(np.arange(len(test_generator.class_indices)))
    ax.set_yticks(np.arange(len(test_generator.class_indices)))
    ax.set_xticklabels(test_generator.class_indices.keys(), rotation=45)
    ax.set_yticklabels(test_generator.class_indices.keys())

plt.show()

# Optional: Save the final models
vgg_model.save('final_vgg16_model.keras')
resnet_model.save('final_resnet50_model.keras')

""""In the dental project, ResNet50 is employed as a sophisticated neural network to classify dental images into
categories such as cavities, implants, or other procedures. Due to its deep architecture and innovative design, ResNet50
is fairly skilled at capturing intricate patterns in image data. By combining its advanced training with custom layers,
the model is highly tuned to deliver accurate predictions. As observed in the name, ResNet50 employs 50 custom layers unlike
the 16 contained in VGG16, allowing it to capture the more intricate and complex patterns contained in the dental images, though
this comes with a significantly higher computational cost.  ResNet50 also utilizes skip connections, ensuring efficient training and improved
performance, even with the added complexity of dental images. After both models have completed training, they are evaluated on
the test dataset, which contains images the models have not been privy to. We are then asked to be provided with the accuracy
score (percentage of correct predictions) and loss function (discrepancy between predicted and actual values), to determine how
well both models performed. Confusion matrices are also added to visualize in which instances the models made correct
predictions and others in which they were misclassified. Precision, recall and F1 scores (addressing for false positives
and negatives) are also all able to provide additional insights.""""


In [ ]:
#Data Augmentation and Image Preprocessing
# Set up ImageDataGenerator for training and validation data
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import drive  # Import the drive module
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount("/content/drive")

# Data Augmentation and Image Preprocessing
# Set up ImageDataGenerator for training and validation data
train_datagen = ImageDataGenerator(
    rescale=1./255,                # Normalize the image pixels to [0, 1]
    rotation_range=30,             # Random rotations
    width_shift_range=0.2,         # Random horizontal shifts
    height_shift_range=0.2,        # Random vertical shifts
    shear_range=0.2,               # Shear transformations
    zoom_range=0.2,                # Random zoom
    horizontal_flip=True,          # Random horizontal flips
    fill_mode='nearest'            # Fill missing pixels
)

val_datagen = ImageDataGenerator(rescale=1./255)  # Just normalize for validation

# Define paths to your train and validation directories (adjust accordingly)
train_dir = '/content/drive/My Drive/final project/train'
val_dir = '/content/drive/My Drive/final project/valid'

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # Resize to match the input size for the model (224x224 for ResNet50, VGG16)
    batch_size=32,
    class_mode='categorical'  # Categorical labels (for multi-class classification)
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Visualize some augmented images
def visualize_augmented_images(generator, num_images=5):
    # Get a batch of images from the generator
    images, labels = next(generator)

    # Plot the images
    plt.figure(figsize=(15, 15))
    for i in range(num_images):
        ax = plt.subplot(1, num_images, i+1)
        ax.imshow(images[i])
        ax.axis('off')
    plt.show()

# Visualize augmented images from the training set
visualize_augmented_images(train_generator)

""""By augmenting our data we improve our the size and diversity of our training data (essentially giving it more to work with).
By exposing the training set to variations (rotated, flipped, shifted) of the same image, we reduce overfitting as the model
is able to discern better between the various types of procedures due to the additional information at its disposal. We continue
with preprocessing our validation set. We conclude that no augmentation is required as the validation set should contain the initial,
unaltered dataset to fairly assess the model’s performance. It was within out interest to visualize these transformed images.
Therefore, we also have code to visually display the augmented images to witness the applied changes. With these enhancements
to the images the model is able to derive crucial statistial insights at a higher level of accuracy, ensuring a robust competent model
that avoids overfitting (that could skew the data).""""

In [ ]:
#. Putting It All Together
#Data Augmentation and Image Preprocessing
# Set up ImageDataGenerator for training and validation data
import os
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from google.colab import drive  # Import the drive module
import matplotlib.pyplot as plt

# Mount Google Drive
drive.mount("/content/drive")

# Data Augmentation and Image Preprocessing
# Set up ImageDataGenerator for training and validation data
train_datagen = ImageDataGenerator(
    rescale=1./255,                # Normalize the image pixels to [0, 1]
    rotation_range=30,             # Random rotations
    width_shift_range=0.2,         # Random horizontal shifts
    height_shift_range=0.2,        # Random vertical shifts
    shear_range=0.2,               # Shear transformations
    zoom_range=0.2,                # Random zoom
    horizontal_flip=True,          # Random horizontal flips
    fill_mode='nearest'            # Fill missing pixels
)

val_datagen = ImageDataGenerator(rescale=1./255)  # Just normalize for validation

# Define paths to your train and validation directories (adjust accordingly)
train_dir = '/content/drive/My Drive/final project/train'
val_dir = '/content/drive/My Drive/final project/valid'

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # Resize to match the input size for the model (224x224 for ResNet50, VGG16)
    batch_size=32,
    class_mode='categorical'  # Categorical labels (for multi-class classification)
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Check if the generators found any images:
print("Number of training samples:", train_generator.samples)
print("Number of validation samples:", val_generator.samples)
print("Number of classes:", train_generator.num_classes)  # This should match the number of subdirectories in the directory

# If train_generator.samples or val_generator.samples are 0, then double check the data directory structure.

# Visualize some augmented images
def visualize_augmented_images(generator, num_images=5):
    # Get a batch of images from the generator
    images, labels = next(generator)

    # Plot the images
    plt.figure(figsize=(15, 15))
    for i in range(num_images):
        ax = plt.subplot(1, num_images, i+1)
        ax.imshow(images[i])
        ax.axis('off')
    plt.show()

# Visualize augmented images from the training set
visualize_augmented_images(train_generator)

""""Data Augmentation for visualization The validation check also ensures that the data generators have correctly loaded the
images and labels from the specified directories, providing confirmation that the dataset structure is compatible with the model’s
specifications.""""

In [ ]:
#Train the model RESNET50
import tensorflow as tf
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.layers import GlobalAveragePooling2D, Dense, Dropout, BatchNormalization
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
from tensorflow.keras.preprocessing.image import ImageDataGenerator
import matplotlib.pyplot as plt

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Step 1: Define paths to your data
train_dir = '/content/drive/MyDrive/final project/train'
val_dir = '/content/drive/MyDrive/final project/valid'
test_dir = '/content/drive/MyDrive/final project/test'

# Step 2: Define your ImageDataGenerators
train_datagen = ImageDataGenerator(
    rescale=1./255,  # Normalize pixel values to [0, 1]
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest'
)

val_datagen = ImageDataGenerator(rescale=1./255)

train_generator = train_datagen.flow_from_directory(
    train_dir,
    target_size=(224, 224),  # Resizing images to fit the model input size
    batch_size=32,
    class_mode='categorical'
)

val_generator = val_datagen.flow_from_directory(
    val_dir,
    target_size=(224, 224),
    batch_size=32,
    class_mode='categorical'
)

# Get number of classes
num_classes = train_generator.num_classes

# Step 3: Create Improved Model - Fine-tune ResNet50 with L2 Regularization, Dropout, and BatchNormalization
def create_improved_model(input_shape=(224, 224, 3), num_classes=num_classes):
    base_model = ResNet50(weights='imagenet', include_top=False, input_shape=input_shape)

    # Fine-tune the last few layers of ResNet50 (reduce the number of layers being fine-tuned)
    for layer in base_model.layers[-10:]:  # Unfreeze the last 10 layers (reduced from 20)
        layer.trainable = True

    # Adding layers on top of the base model
    x = GlobalAveragePooling2D()(base_model.output)
    x = Dense(64, activation='relu', kernel_regularizer=l2(0.01))(x)  # Reduced from 128 to 64 units
    x = Dropout(0.3)(x)  # Reduced dropout rate
    output = Dense(num_classes, activation='softmax')(x)  # Output layer for classification

    model = Model(inputs=base_model.input, outputs=output)

    # Compile the model with Adam optimizer and a fixed learning rate
    model.compile(optimizer=Adam(learning_rate=1e-4),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])

    return model

# Step 4: Train and Evaluate the Model
def train_and_evaluate(model, train_generator, val_generator, epochs=7):  # Change epochs to 7
    # Training the model
    history = model.fit(train_generator,
                        epochs=epochs,
                        validation_data=val_generator)

    # Plot Training & Validation Accuracy and Loss
    plot_training_history(history)

    # Evaluate on Test Data
    test_datagen = ImageDataGenerator(rescale=1./255)
    test_generator = test_datagen.flow_from_directory(test_dir, target_size=(224, 224), batch_size=32, class_mode='categorical')

    test_loss, test_accuracy = model.evaluate(test_generator)
    print(f"Test Loss: {test_loss}")
    print(f"Test Accuracy: {test_accuracy}")

# Step 5: Plot Training History
def plot_training_history(history):
    # Plot Training & Validation Accuracy and Loss
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Accuracy plot
    ax1.plot(history.history['accuracy'], label='Training Accuracy')
    ax1.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax1.set_title('Training and Validation Accuracy')
    ax1.set_xlabel('Epochs')
    ax1.set_ylabel('Accuracy')
    ax1.legend()

    # Loss plot
    ax2.plot(history.history['loss'], label='Training Loss')
    ax2.plot(history.history['val_loss'], label='Validation Loss')
    ax2.set_title('Training and Validation Loss')
    ax2.set_xlabel('Epochs')
    ax2.set_ylabel('Loss')
    ax2.legend()

    plt.show()

# Step 6: Train the Model with Reduced Layers
model = create_improved_model()
print("Training Improved Model...")
train_and_evaluate(model, train_generator, val_generator, epochs=7)  # Training for 7 epochs
""""We begin again with data augmentation to increase the diversity of our dataset that will help mitigate overfitting. After
pre-processing, we focused on creating a more refined model of ResNet50 that contained enhancements through fine tuning. To begin,
only the last 10 layers of ResNet50 are unfrozen, allowing fine-tuning while keeping most of the pre-trained weights intact,
balancing computational cost with accuracy. Our dropout also increased to deactivates 30% of neurons randomly during training to
improve generalization and refine the model's capacity for accurately interpreting and predicting outcomes on unencountered data.
Again we apply Adam Optimize and Loss Function to further enhance the model. The model is then trained and evaluated in three stages:
training, visualization and testing (using 7 epochs this time).
""""

In [ ]:
#Image classification in 4 categories
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(128, 128, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(128, activation='relu'),
    Dropout(0.5),
    Dense(4, activation='softmax')  # 4 classes
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()


""""This model is a Convolutional Neural Network (CNN) designed to classify images into four categories. It begins with convolutional
layers that capture geometric patterns and layered features, utlizing ReLU activation to capture complex patterns discovered in
the dataset. Max pooling layers follow to reduce dimensionality and focus on the most prominent features. The feature maps are
then flattened and passed through dense layers to refine classification, with dropout added to prevent overfitting. The final output
layer uses softmax activation to assign probabilities to each category with respect to image classification. The model is
compiled with Adam optimize for efficient training, categorical crossentropy as the loss function and accuracy to evaluate
the overall performance of the newly enhanced and robust ResNet50 model."""

In [ ]:
"""In conclusion, This project aimed to classify dental images into specific categories using advanced deep learning models like VGG16 and ResNet50.
Through data augmentation, preprocessing, and fine-tuning of pre-trained networks, the models were trained to capture essential features from the dataset
while ensuring robust generalization. ResNet50, with its deeper architecture and fine-tuned layers, demonstrated an
enhanced ability to distinguish subtle variations in the images compared to VGG16. """